# Hausse de loyer Équinoxe : notebook de départ

**Votre tâche : estimer la hausse de loyer d'Équinoxe pour 2026.**

Ce notebook vous aide à vous orienter et vous montre une méthode correcte. Les prévisions, les données externes et les décisions de jugement
vous reviennent. Si c'est la première fois que vous utilisez un notebook, pas d'inquiétude. C'est un outil très simple à utiliser et puissant, qui rend les flux de travail clairs et faciles à communiquer. Il est couramment utilisé
en apprentissage profond, et le maîtriser vous aidera énormément si vous vous dirigez un jour vers la science des données ou tout autre domaine analytique. **CTRL/COMMAND + MAJ + H** affiche les raccourcis. Apprenez à supprimer, restaurer, créer et exécuter des cellules. Ça vous aidera.

Lisez les cellules, ne vous contentez pas de les exécuter.

#### Important : vous verrez que la description des données ne correspond pas toujours parfaitement à ce qu'elles sont réellement. Dans ce cas-ci, PromoPay dans les concessions est décrit comme un frais mensuel, ce qui n'est pas du tout le cas. C'est assez facile à comprendre : si la promo égalait le loyer, le résident habiterait en pratique gratuitement.
##### Votre travail consiste aussi à découvrir ce que les données représentent réellement, et à les renommer selon votre propre description. Il n'y a pas de réponse unique.

## 1. Charger les données

## 1b. Lire les colonnes : champs `h` et champs `s`

Chaque colonne de ce jeu commence par **`h`** ou **`s`**. C'est la convention de Yardi, et
se tromper là-dessus est la façon la plus courante de corrompre une requête de gestion immobilière.

| préfixe | signification | exemple | à quoi ça sert |
|---|---|---|---|
| **`h`** | **handle** (identifiant) - une clé numérique | `hUnit`, `hProperty`, `hBuilding` | **les jointures**. Opaque, stable, unique. |
| **`s`** | **stored value** (valeur stockée) - ce qu'un humain lit | `sUnitCode`, `sRent`, `sBuilding` | **l'affichage et le regroupement**. |

Yardi conserve les deux moitiés d'une même information côte à côte - `hAgent` à côté de `sAgent`,
`hProperty` à côté de `sPropCode`. Elles ne sont **pas interchangeables** :

> **Joindre sur `h`. Étiqueter avec `s`.**

Ce n'est pas du pinaillage, c'est tout l'objet de la section 4 ci-dessous. `sUnitCode` vaut `"0203"` - et
trois appartements différents sur un même site portent cette même chaîne. `hUnit` est unique par
construction. Si vous associez une unité à elle-même du côté `s`, vous fusionnez des appartements sans le savoir;
si vous associez sur `hUnit`, c'est impossible.

**Deux mises en garde sur `s`, qui reflètent toutes deux le vrai comportement de Yardi, pas des particularités de ce jeu :**

1. **`s` ne veut pas dire « string ».** La table `UNIT` de Yardi stocke `SRENT` et `DSQFT` comme
   *valeurs numériques*. Ici, `sRent` et `sSqft` sont des nombres. Ne déduisez pas un type d'un préfixe -
   vérifiez-le.
2. **Le vrai Yardi subdivise davantage le côté `s`** - `dt*` pour les dates (`dtLeaseFrom`), `d*`
   pour les doubles (`dSqft`), `i*` pour les entiers, `b*` pour les indicateurs. Ce jeu les ramène tous
   à `s` pour garder l'attention sur la distinction `h` / `s`. Quand vous ouvrirez la vraie base de données, attendez-vous
   à un alphabet plus long.

### Les codes de frais

`sChargeCode` dans le fichier des concessions correspond au plan comptable de Yardi. Les codes semblent abîmés
parce que `sCode` est un `nchar(8)` - « Free other » devient `Freeothe`, « Referencing » devient
`Referenc`. Huit caractères, tronqués, avec une casse incohérente. C'est à ça que ressemblent les vrais.

| `sChargeCode` | `sChargeDesc` | ce que c'est réellement |
|---|---|---|
| `PromoPay` | Promo - Payable in the future | Un rabais promotionnel. **Lisez la mise en garde ci-dessous.** |
| `freerent` | Free rent | Loyer annulé pour une partie du bail. |
| `freepark` | Free Parking | Frais de stationnement annulés. Un avantage *accessoire*, pas du loyer. |
| `freelock` | Free Locker | Casier de rangement offert. Accessoire. |
| `freeappl` | Free Appliances | Frais d'électroménagers annulés. Accessoire. |
| `Freeothe` | Free other | Tout ce qui n'est pas catégorisé. À traiter avec méfiance. |
| `Indem` | Client indemnity | Un paiement versé au résident, par exemple pour régler un différend. |
| `Referenc` | Referencing program | Un crédit de référencement. |

**Celui qu'il faut examiner avant de s'y fier :** `PromoPay` est décrit comme un frais
mensuel. Ce n'en est pas un. Il est très majoritairement inscrit sur un seul mois, pour un montant proche d'un mois
de loyer complet. Prenez la description au pied de la lettre et vous conclurez que les résidents habitent
presque gratuitement. Comparez `sMonths` et la taille de `sAmount` à `sRent` avant de le modéliser -
et décidez vous-même comment l'étaler sur la durée du bail.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

listings   = pd.read_csv("equinoxe_listings.csv")
leases     = pd.read_csv("equinoxe_lease_history.csv")
concessions= pd.read_csv("equinoxe_concessions.csv")
asking     = pd.read_csv("equinoxe_asking_history.csv")

for name, df in [("listings", listings), ("leases", leases),
                 ("concessions", concessions), ("asking", asking)]:
    print(f"{name:<12} {len(df):>6,} lignes   {df.shape[1]:>2} colonnes")

In [ ]:
# Convertir les dates une seule fois, dès le départ.
for col in ["sLeaseFrom", "sLeaseTo", "sSignDate", "sAvailable"]:
    if col in leases.columns:
        leases[col] = pd.to_datetime(leases[col], errors="coerce")

leases["year"] = leases.sLeaseFrom.dt.year
asking["date"] = pd.to_datetime(asking.sMonth + "-01")

print(leases.year.value_counts().sort_index().to_string())

## 2. Ce que vous avez sous les yeux

Six immeubles répartis entre Laval, Mont-Royal, Ottawa et Pointe-Claire. Tout est en date du **2025-12-31**;
il n'y a pas de 2026 dans ce jeu, parce que 2026 est justement ce que vous prédisez.

Cinq sont au Québec et un, The Met, est en Ontario. Ce n'est pas
un détail : les deux provinces encadrent les hausses de loyer selon des régimes différents, et
les traiter comme un seul portefeuille est un choix de modélisation que vous devrez défendre.

**Une chose à bien comprendre avant de commencer.** Il s'agit d'un *extrait publié*, pas du
portefeuille. Il contient une part fixe des unités de chaque immeuble et de chaque type de chambre - la
répartition est représentative, les volumes ne le sont pas. Donc tout ce qui nécessite un dénominateur d'unités
réelles est hors de portée ici : **l'occupation, l'inoccupation, l'absorption et le rôle de loyers total du portefeuille
ne peuvent pas être calculés à partir de ce jeu, et un chiffre que vous en tireriez serait un artefact de
l'extrait plutôt qu'un fait sur l'entreprise.** C'est la *variation* des loyers que ces données permettent de mesurer,
et c'est la variation des loyers qu'on vous demande.

In [ ]:
print(listings.groupby(["sState", "sCity", "sBuilding"]).size().to_string())
print()
print(listings.groupby(["sBeds", "sBaths"])
      .agg(units=("sRent", "size"),
           median_rent=("sRent", "median"),
           median_sqft=("sSqft", "median"))
      .to_string())

## 3. L'approche évidente

Le premier réflexe naturel : le loyer médian par année.

In [ ]:
naive = (leases[leases.year.between(2019, 2025)]
         .groupby("year")
         .agg(leases=("sRent", "size"), median_rent=("sRent", "median")))
naive["yoy_pct"] = (naive.median_rent.pct_change() * 100).round(2)
naive

Selon ce tableau, les loyers auraient augmenté de **10,8 % en 2023**. Ce n'est pas le cas.

La médiane bouge dès que la *composition* bouge, et la composition a beaucoup bougé. Plusieurs
immeubles ont été mis en service en 2023 et 2024. Regardez ce qui se passe sous la
médiane :

In [ ]:
window = leases[leases.year.between(2019, 2025)]
mix = window.groupby("year").agg(
    leases=("sRent", "size"),
    median_sqft=("sSqft", "median"),
    pct_2bed_plus=("sBeds", lambda s: (s >= 2).mean() * 100),
    buildings=("sBuilding", "nunique"))
mix["rent_per_sqft"] = (window.assign(psf=window.sRent / window.sSqft)
                        .groupby("year").psf.median())
mix.round(2)

In [ ]:
# À partir de quand chaque sBuilding a-t-il commencé à générer des baux?
print(window.groupby("sBuilding").year.min().sort_values().to_string())
print()
# Et combien demandent-ils? (baux de 2024)
print(window[window.year == 2024].groupby("sBuilding")
      .agg(leases=("sRent", "size"), median_rent=("sRent", "median"))
      .sort_values("median_rent").to_string())

Les unités louées en 2024 sont **plus petites** que celles louées en 2021 : la superficie
médiane est passée de 1 175 à 1 045 pi², et pourtant le loyer médian a fortement augmenté. C'est
le contraire de ce que la taille seule laisserait prévoir. La part des baux de 2 chambres et plus
est passée de 78 % à 62 % sur la même période.

Ce qui s'est réellement passé : Le Carlyle, The Met et Westpark ont été mis en service en 2023 et 2024. Ce sont des
immeubles haut de gamme, qui louent des unités plus petites, davantage axées sur les 1 chambre, à des tarifs au pied carré
bien plus élevés - la médiane 2024 de The Met est de 2 752 $, contre 1 672 $ pour Lévesque.
Le loyer au pied carré est passé de 1,71 $ à 2,20 $ en deux ans, et presque rien de
cela ne vient d'un locataire existant qui paie plus.


## 4. La méthode qui fonctionne : comparer une unité à elle-même

Associez les baux consécutifs de chaque unité et annualisez la variation. Même appartement,
même immeuble, tout pareil : ce qui reste, c'est le prix.

**La seule chose à ne pas rater :** associez sur `sPropCode` + `sUnitCode`.
Pas sur `sSite` + `sUnitCode`. Plusieurs sites regroupent plus d'un code de propriété avec
des numéros d'unité qui se chevauchent, et **248 des 1 061 unités entrent en collision** si vous utilisez
le site comme clé. Exécutez la cellule suivante pour le constater, puis utilisez `sPropCode` + `sUnitCode`, ou
la colonne `sLink`, qui est construite sur la clé sûre.

In [ ]:
# Pourquoi sSite + sUnitCode n'est pas une clé : un site, un numéro d'unité, plusieurs appartements.
# (hUnit est l'identifiant qui EST unique - voir le dictionnaire de données plus haut.)
dupes = listings[listings.duplicated(["sSite", "sUnitCode"], keep=False)]
print(f"{len(dupes):,} unités sur {len(listings):,} entrent en collision sur sSite + sUnitCode")
print()
print("Nombre de codes de propriété (sPropCode) par site :")
print(listings.groupby("sSite").sPropCode.nunique().to_string())

In [ ]:
UNIT_KEY = ["sPropCode", "sUnitCode"]      # <-- pas ["sSite", "sUnitCode"]

def same_unit_growth(df, price_col="sRent", date_col="sLeaseFrom",
                     min_gap=0.5, max_gap=2.5):
    """Variation annualisée du loyer, en comparant chaque unité à son propre bail précédent.

    Conserve les écarts entre `min_gap` et `max_gap` années : plus court signifie
    généralement le même bail enregistré deux fois, plus long est trop ancien pour une comparaison équivalente.
    """
    d = df.sort_values(UNIT_KEY + [date_col]).copy()
    d["prev_price"] = d.groupby(UNIT_KEY)[price_col].shift(1)
    d["prev_date"]  = d.groupby(UNIT_KEY)[date_col].shift(1)
    d = d.dropna(subset=["prev_price", "prev_date"])

    d["gap_years"] = (d[date_col] - d.prev_date).dt.days / 365.25
    d = d[d.gap_years.between(min_gap, max_gap, inclusive="neither")]
    d = d[(d.prev_price > 0) & (d[price_col] > 0)]

    d["growth_pct"] = ((d[price_col] / d.prev_price) ** (1 / d.gap_years) - 1) * 100
    return d

pairs = same_unit_growth(leases)
print(f"{len(pairs):,} paires utilisables sur {pairs.groupby(UNIT_KEY).ngroups:,} unités")

honest = (pairs[pairs.year.between(2021, 2025)]
          .groupby("year")
          .growth_pct.agg(pairs="size", median="median").round(2))
honest

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
yrs = range(2021, 2026)
ax.plot(yrs, naive.loc[yrs, "yoy_pct"], "o--", label="naïf : médiane de tous les baux", lw=2)
ax.plot(yrs, honest.loc[yrs, "median"], "o-", label="même unité comparée à elle-même", lw=2)
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("croissance annuelle des loyers (%)")
ax.set_title("La méthode naïve mesure surtout la composition")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



## 5. Loyer contractuel n'est pas loyer perçu

`sRent` est ce que le bail indique. `sRentEffective` est ce qu'Équinoxe a réellement
perçu après les mois gratuits, le stationnement gratuit et les casiers gratuits. Les concessions sont maintenant
la norme, et elles progressent.

In [ ]:
conc = (leases[leases.year.between(2021, 2025)]
        .groupby("year")
        .agg(leases=("sRent", "size"),
             pct_with_concession=("sConcession", lambda s: s.mean() * 100),
             median_contracted=("sRent", "median"),
             median_effective=("sRentEffective", "median")))
conc["gap_pct"] = ((conc.median_effective / conc.median_contracted - 1) * 100)
conc.round(2)

In [ ]:
# La concession change-t-elle l'évolution, ou seulement le niveau?
eff = same_unit_growth(leases, price_col="sRentEffective")
compare = pd.DataFrame({
    "contracted": honest["median"],
    "effective":  eff[eff.year.between(2021, 2025)].groupby("year").growth_pct.median(),
}).round(2)
compare

Ces deux colonnes évoluent de près jusqu'en 2023 - à moins
d'un point l'une de l'autre - puis elles divergent : en 2025, la croissance contractuelle est de
7,0 % alors que la croissance effective est de 3,6 %. L'écart de *niveau* du tableau précédent
se creuse tout au long de la période.

Elles répondent à des questions différentes, et l'année où elles se séparent est celle où les concessions
deviennent plus importantes plutôt que simplement plus fréquentes. Si vous pouvez expliquer ce que cette séparation signifie pour un
propriétaire - et laquelle des deux vous citeriez à un propriétaire - vous comprenez les données
mieux que la plupart des équipes.

## 6. Les renouvellements ne se comportent pas comme les relocations

`sRenewal = 1`, c'est un locataire en place qui renouvelle. `sRenewal = 0`, c'est une unité
qui se libère et qui est relouée au prix du marché. Au Québec, ces deux cas sont régis par
des règles différentes, et l'écart entre les deux est toute la raison pour laquelle un propriétaire s'intéresse
à ce chiffre. Remarquez que l'écart change de camp au cours de la période : les renouvellements mènent en 2022, les relocations en 2025.

In [ ]:
split = (pairs[pairs.year.between(2022, 2025)]
         .groupby(["year", "sRenewal"])
         .growth_pct.agg(pairs="size", median="median").round(2)
         .unstack("sRenewal"))
split.columns = [f"{a}_{'renewal' if b else 'turnover'}" for a, b in split.columns]
split

## 7. À vous de jouer

Tout ce qui précède est descriptif. Rien de tout ça n'est une prévision, et 2026 n'est pas dans
ce jeu. Ce qui reste, c'est le vrai travail :

**a) Décidez à quelle question vous répondez.** « La hausse de loyer de l'année », ce sont en fait
plusieurs chiffres différents : même unité contractuel, même unité effectif, renouvellements
seulement, relocations seulement, portefeuille combiné. Ils ne concordent pas. Choisissez-en un, nommez-le dans
votre rapport et justifiez votre choix.

**b) Intégrez des données publiques.**

- **Enquête sur les logements locatifs de la SCHL** : loyers et taux d'inoccupation par nombre de chambres pour les
  RMR de Montréal et d'Ottawa, ainsi que la répartition entre logements ayant changé de locataire ou non, qui correspond
  directement à `sRenewal`
- **IPC de Statistique Canada**, composante loyers, Québec et Ontario
- **Tribunal administratif du logement** : les paramètres annuels d'augmentation de loyer au Québec.
  Cinq des six immeubles y sont assujettis
- **Taux légal d'augmentation des loyers en Ontario** : s'applique à l'immeuble The Met, selon
  des règles différentes. N'appliquez pas le plafond d'une province à l'autre

Utilisez la série externe sur l'inoccupation comme *contexte* pour l'orientation des loyers. N'essayez pas
de la transposer en taux d'occupation propre à ce portefeuille - voir la note de la section 2.

**c) Faites une prévision.** Une extrapolation de tendance est un point de départ légitime. Une approche
qui concilie la tendance interne avec le marché externe est meilleure. Énoncez
vos hypothèses.

**e) Misez sur le raisonnement et faites de votre mieux.** Si votre méthode atteint une précision de 95 % et que quelqu'un d'autre obtient 96 %, ça ne veut pas dire que l'autre a gagné. Vous êtes évalué davantage sur la technique que sur le résultat. Les résultats comptent, mais sans un raisonnement constamment démontré derrière chaque choix, même 99 % ne veut rien dire. Nous devons voir une logique claire derrière vos décisions. Note : vous devez **TOUJOURS** expliquer une étape importante, même si ça vous semble inutile ou peu naturel.

## 7.1 Définition retenue

**Nous estimons la croissance annualisée à unité constante du loyer effectif (`sRentEffective`) pour 2026**, combinée par segment selon la province, le régime réglementaire (TAL, section F, Ontario) et le type de bail (renouvellement ou relocation), soit 5 segments (section 7.7).

- **Unité constante** : chaque appartement est comparé à son propre bail précédent, sur la clé `sPropCode` + `sUnitCode` (section 4). Cela élimine l'effet de composition (section 3).
- **Loyer effectif** : c'est le revenu réellement perçu par le propriétaire. En 2025, la hausse contractuelle même unité (7,04 %) était environ deux fois plus forte que la hausse effective (3,58 %), à cause des concessions (section 5).
- **Par segment** : les renouvellements et les relocations obéissent à des règles différentes (section 6), les immeubles neufs du Québec relèvent de la section F, et The Met relève de l'Ontario (section 7.5).
- **Combinaison** : chaque segment est pondéré par sa part dans les paires de l'année précédente. Il s'agit d'une part de paires, pas d'un taux d'occupation, que cet extrait ne permet pas de calculer (section 2).

## 7.2 Ce que représentent vraiment les données

Les codes de frais de `concessions` sont tronqués à 8 caractères et leurs descriptions sont parfois trompeuses. Nous les renommons selon ce qu'ils représentent réellement, puis nous vérifions le cas de `PromoPay`.

In [ ]:
CHARGE_CODE_LABELS = {
    "PromoPay": "rabais_promo_unique",
    "freerent": "rabais_loyer_mensuel",
    "freepark": "stationnement_gratuit",
    "freelock": "casier_gratuit",
    "freeappl": "electromenagers_gratuits",
    "Freeothe": "autre_gratuite",
    "Indem":    "indemnite_mensuelle",
    "Referenc": "credit_referencement",
}
SINGLE_MONTH = 1

concessions["concession_type"] = concessions.sChargeCode.map(CHARGE_CODE_LABELS)

concession_profile = (concessions.groupby("concession_type")
                      .agg(nb_lignes=("sAmount", "size"),
                           montant_median=("sAmount", "median"),
                           mois_median=("sMonths", "median"),
                           pct_un_seul_mois=("sMonths", lambda s: (s == SINGLE_MONTH).mean() * 100))
                      .sort_values("nb_lignes", ascending=False)
                      .round(2))
print(concession_profile.to_string())
print(f"\nLoyer contractuel médian des baux (pour comparaison) : {leases.sRent.median():.0f} $")

### Interprétation

Les montants sont négatifs car ce sont des crédits accordés au locataire. Deux codes ne correspondent pas à leur description :

1. **`PromoPay` (« Promo - Payable in the future ») est un rabais unique, pas un frais mensuel.** 88 % des lignes portent sur un seul mois, pour un montant médian de 1 357 $, soit environ 69 % du loyer contractuel médian (1 970 $). Pris comme un rabais mensuel, il laisserait croire que les résidents habitent presque gratuitement. Nous le renommons `rabais_promo_unique` et le considérons comme un rabais ponctuel à étaler sur la durée du bail (environ 113 $/mois sur 12 mois).

2. **`freerent` (« Free rent ») n'est pas un mois gratuit, mais un rabais mensuel récurrent** : 12 mois à environ 129 $/mois. Nous le renommons `rabais_loyer_mensuel`. De même, `Indem` n'est pas un paiement unique mais un crédit mensuel (`indemnite_mensuelle`).

Tous les autres codes sont des crédits mensuels étalés sur 12 mois (environ 130 à 155 $/mois).

**Conséquence pour le modèle :** nous utilisons `sRentEffective`, qui intègre ces concessions dans un loyer mensuel moyen. Pour vérifier qu'il étale bien `PromoPay`, nous avons contrôlé des baux individuels : l'écart entre loyer effectif et loyer contractuel correspond à environ un mois de rabais réparti sur un bail de 12 mois, et non à un rabais mensuel complet.

## 7.3 L'effet de composition

La médiane annuelle compare des appartements différents d'une année à l'autre. Entre 2022 et 2023, le nombre d'immeubles actifs passe de 3 à 5 (Le Carlyle et The Met), puis à 6 en 2024 (Westpark). Ces immeubles haut de gamme louent des unités plus petites mais beaucoup plus chères au pied carré. La médiane affiche +10,85 % en 2023, alors que la hausse même unité n'est que de 2,29 %. En 2025, l'erreur s'inverse (4,06 % naïf contre 7,04 % même unité) : la médiane annuelle n'est pas biaisée dans un sens fixe, elle est simplement non fiable.

In [ ]:
ANALYSIS_YEARS = range(2021, 2026)

fig, (ax_growth, ax_profile) = plt.subplots(1, 2, figsize=(14, 4.5))

ax_growth.plot(ANALYSIS_YEARS, naive.loc[ANALYSIS_YEARS, "yoy_pct"], "o--", lw=2,
               label="Médiane annuelle (naïve)")
ax_growth.plot(ANALYSIS_YEARS, honest.loc[ANALYSIS_YEARS, "median"], "s-", lw=2,
               label="Même unité (contractuel)")
ax_growth.set_title("La médiane annuelle mesure surtout la composition")
ax_growth.set_ylabel("hausse annuelle (%)")
ax_growth.legend()
ax_growth.grid(alpha=0.3)

ax_rent_psf = ax_profile.twinx()
ax_profile.plot(mix.index, mix.median_sqft, "v-", color="tab:green", lw=2, label="Superficie médiane (pi²)")
ax_rent_psf.plot(mix.index, mix.rent_per_sqft, "d-", color="tab:orange", lw=2, label="Loyer médian ($/pi²)")
ax_profile.set_title("Les unités louées rapetissent, le prix au pi² explose")
ax_profile.set_ylabel("superficie (pi²)")
ax_rent_psf.set_ylabel("loyer ($/pi²)")
ax_profile.legend(loc="upper left")
ax_rent_psf.legend(loc="upper right")
ax_profile.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7.4 Les concessions : pourquoi le loyer contractuel trompe le propriétaire

La part des baux avec concession passe de 54 % en 2021 à 85 % en 2025, et l'écart médian entre loyer effectif et contractuel passe de −3,5 % à −8,7 %. Jusqu'en 2023, les hausses même unité contractuelle et effective restent à moins d'un point l'une de l'autre. En 2024-2025, elles divergent : en 2025, +7,04 % en contractuel contre +3,58 % en effectif. Environ la moitié de la hausse affichée est redonnée en concessions. Les concessions deviennent à la fois plus fréquentes et plus importantes, ce qui confirme notre choix du loyer effectif.

In [ ]:
concession_gap = compare.assign(ecart_concessions_pts=(compare.contracted - compare.effective).round(2))
print(concession_gap.to_string())

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(compare.index, compare.contracted, "o-", lw=2, label="Loyer contractuel (sRent)")
ax.plot(compare.index, compare.effective, "s-", lw=2, label="Loyer effectif (sRentEffective)")
ax.fill_between(compare.index, compare.contracted, compare.effective, alpha=0.25,
                label="Hausse absorbée par les concessions")
ax.set_title("Hausse même unité : contractuel vs effectif")
ax.set_ylabel("hausse annuelle (%)")
ax.set_xticks(list(compare.index))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7.5 Renouvellements, relocations et règles applicables

Les hausses ne sont pas encadrées de la même façon selon la province, le type de bail et l'âge de l'immeuble :

| Régime | Immeubles concernés | Règle pour un locataire qui renouvelle |
|---|---|---|
| **Québec, encadré par le TAL** | Immeubles de plus de 5 ans | Le locataire peut refuser la hausse et le TAL peut fixer le loyer. Le taux de référence du TAL sert de point d'ancrage. |
| **Québec, section F (immeuble neuf)** | Immeubles prêts pour l'habitation depuis 5 ans ou moins | Si la section F du bail est remplie, le locataire ne peut pas demander au TAL de fixer le loyer : s'il refuse la hausse, il doit quitter. Les hausses suivent donc le marché. |
| **Ontario, exempté** | Logements occupés pour la première fois après le 15 novembre 2018 | La ligne directrice provinciale ne s'applique pas. |
| **Ontario, ligne directrice** | Logements occupés au plus tard le 15 novembre 2018 | La hausse est plafonnée par la ligne directrice (2,1 % en 2026). |

Pour les **relocations**, le prix se rapproche du marché dans les deux provinces.

Nous déduisons l'âge de chaque immeuble de la date de son **premier bail dans nos données**. C'est une approximation : la date officielle de mise en habitation devrait être confirmée auprès d'Équinoxe.

In [ ]:
QUEBEC_LABEL = "Quebec"
ONTARIO_LABEL = "Ontario"
ONTARIO_EXEMPTION_CUTOFF = pd.Timestamp("2018-11-15")
QUEBEC_SECTION_F_MAX_AGE_YEARS = 5
FORECAST_START = pd.Timestamp("2026-01-01")
REGIME_ANALYSIS_START_YEAR = 2023

REGIME_QC_TAL = "QC_encadre_TAL"
REGIME_QC_SECTION_F = "QC_section_F_neuf"
REGIME_ON_EXEMPT = "ON_exempte_post_2018"
REGIME_ON_GUIDELINE = "ON_ligne_directrice"


def regulatory_regime(row):
    """Régime applicable aux renouvellements d'un immeuble, selon sa province et son âge."""
    if row.sState == ONTARIO_LABEL:
        return REGIME_ON_EXEMPT if row.premier_bail > ONTARIO_EXEMPTION_CUTOFF else REGIME_ON_GUIDELINE
    building_age_years = (FORECAST_START - row.premier_bail).days / DAYS_PER_YEAR
    return REGIME_QC_SECTION_F if building_age_years <= QUEBEC_SECTION_F_MAX_AGE_YEARS else REGIME_QC_TAL


DAYS_PER_YEAR = 365.25

building_regimes = (leases.groupby(["sBuilding", "sState"]).sLeaseFrom.min()
                    .reset_index(name="premier_bail"))
building_regimes["regime"] = building_regimes.apply(regulatory_regime, axis=1)
print(building_regimes.to_string(index=False))

growth_by_regime = (eff.merge(building_regimes[["sBuilding", "regime"]], on="sBuilding")
                    .loc[lambda d: d.year >= REGIME_ANALYSIS_START_YEAR]
                    .assign(type_bail=lambda d: np.where(d.sRenewal.eq(1), "renouvellement", "relocation"))
                    .groupby(["regime", "type_bail", "year"])
                    .growth_pct.agg(paires="size", mediane="median").round(2)
                    .unstack("year"))
print()
print("Hausse même unité du loyer effectif, par régime et type de bail :")
print(growth_by_regime.to_string())

## 7.6 Données externes

| Donnée | 2022 | 2023 | 2024 | 2025 | 2026 | Source |
|---|---|---|---|---|---|---|
| Taux de base TAL, logement non chauffé (%) | 1,28 | 2,3 | 4,0 | 5,9 | 3,1 | Communiqués « fixation » du TAL ; Le Devoir (janv. 2026) |
| Ligne directrice Ontario (%) | — | 2,5 | 2,5 | 2,5 | 2,1 | ontario.ca, ligne directrice sur l'augmentation des loyers |
| Hausse du loyer moyen 2 chambres, RMR de Montréal (%) | 5,4 | 7,9 | 6,3 | 7,2 | — | SCHL, Rapport sur le marché locatif |
| Hausse du loyer moyen 2 chambres, RMR d'Ottawa (%) | — | — | 5,0 | 3,4 | — | SCHL, Rapport sur le marché locatif |

**Contexte de marché (SCHL, 2025)** : à Montréal, l'inoccupation passe de 2,1 % à 2,9 %, et atteint 4,9 % dans le quartile des loyers les plus chers. À Ottawa, elle atteint 3,0 %, et 6,7 % dans les logements neufs. Ces taux servent uniquement de contexte : aucun taux d'occupation n'est calculé à partir de l'extrait d'Équinoxe.

**Validation croisée** : à Montréal, en 2023, la SCHL mesure une hausse de 18,9 % pour un 2 chambres qui change de locataire, contre 5,7 % quand le locataire reste. Le marché confirme l'écart entre relocations et renouvellements.

**Règle anti-fuite** : pour prévoir l'année N, nous n'utilisons que l'information publiée avant N. Le taux du TAL est publié en janvier, donc il est utilisable pour l'année N. Le rapport de la SCHL est publié en décembre : pour prévoir N, nous utilisons celui de N−1.

**Note sur le TAL 2025** : avec la nouvelle méthode, le TAL a recalculé à 4,5 % le taux applicable aux baux d'avril 2025 à avril 2026. Nous gardons 5,9 %, le taux publié en janvier 2025, qui était en vigueur lorsque les renouvellements de 2025 ont été négociés.

In [ ]:
TAL_BASE_PCT = {2022: 1.28, 2023: 2.3, 2024: 4.0, 2025: 5.9, 2026: 3.1}
ONTARIO_GUIDELINE_PCT = {2023: 2.5, 2024: 2.5, 2025: 2.5, 2026: 2.1}
SCHL_RENT_GROWTH_PCT = {
    "QC": {2022: 5.4, 2023: 7.9, 2024: 6.3, 2025: 7.2},
    "ON": {2022: None, 2023: None, 2024: 5.0, 2025: 3.4},
}

external_summary = pd.DataFrame({
    "tal_base_pct": TAL_BASE_PCT,
    "ontario_ligne_directrice_pct": ONTARIO_GUIDELINE_PCT,
    "schl_montreal_pct": SCHL_RENT_GROWTH_PCT["QC"],
    "schl_ottawa_pct": SCHL_RENT_GROWTH_PCT["ON"],
}).sort_index()
print(external_summary.to_string())

## 7.7 Modèle de prévision et backtest

**Segments.** Chaque paire même unité est classée dans l'un de 5 segments selon sa province, son régime (section 7.5) et son type de bail. Chaque segment est ancré sur la donnée externe qui le gouverne :

| Segment | Ancre externe |
|---|---|
| Renouvellements Québec encadrés par le TAL | Taux de base du TAL de l'année |
| Renouvellements Québec en section F | Croissance SCHL Montréal de l'année précédente |
| Relocations Québec | Croissance SCHL Montréal de l'année précédente |
| Renouvellements The Met (exempté) | Croissance SCHL Ottawa de l'année précédente |
| Relocations The Met | Croissance SCHL Ottawa de l'année précédente |

**Méthode, pour chaque segment :**
1. **Hausse contractuelle** = poids_ancre × (ancre externe + poids_écart × écart historique moyen d'Équinoxe face à l'ancre) + (1 − poids_ancre) × tendance interne des 3 dernières années. On travaille en contractuel parce que le TAL et l'Ontario encadrent le loyer du bail, pas les rabais.
2. **Effet des concessions** = écart entre croissance effective et contractuelle, selon trois scénarios : *bas* (les concessions grossissent autant qu'en 2025), *central* (au rythme moyen des 3 dernières années), *haut* (elles se stabilisent).
3. **Hausse effective** = hausse contractuelle + effet des concessions.

Les segments sont ensuite combinés selon leur part dans les paires de l'année précédente.

**Sélection des réglages par backtest.** Nous testons 9 combinaisons de `poids_écart` et `poids_ancre` (0 ; 0,5 ; 1) et retenons celle qui minimise l'erreur absolue moyenne sur 2023-2025, comparée à deux modèles de référence : la hausse de l'année précédente et la moyenne sur 3 ans. Chaque prévision n'utilise que les données antérieures à l'année prévue.

**Limites.** Les réglages sont choisis sur les mêmes années que le backtest : l'erreur affichée est donc légèrement optimiste. Les segments section F et The Met n'ont que 2 années d'historique. La table `asking` (loyers affichés) n'est pas utilisée dans cette version ; l'écart entre loyer affiché et loyer en place serait une piste d'amélioration pour les relocations.

**Note sur le « réel » du backtest :** il combine les médianes de chaque segment, pondérées par leur part (4,02 % en 2025). Il diffère de la médiane globale de toutes les paires (3,58 % en 2025, section 5), car la médiane d'un mélange n'est pas le mélange des médianes. Nous utilisons la même construction pour la prévision et pour le réel, afin que la comparaison soit juste.

In [ ]:
LOOKBACK_YEARS     = 3
FORECAST_YEAR      = 2026
BACKTEST_YEARS     = (2023, 2024, 2025)
WEIGHT_GRID        = (0.0, 0.5, 1.0)
DRAG_SCENARIOS     = ("bas", "central", "haut")
CENTRAL_SCENARIO   = "central"
CONTRACT_PRICE_COL = "sRent"
TARGET_PRICE_COL   = "sRentEffective"
RENEWAL_FLAG       = 1

SEG_QC_TAL_RENEWAL       = "QC_TAL_renouvellement"
SEG_QC_SECTION_F_RENEWAL = "QC_sectionF_renouvellement"
SEG_QC_TURNOVER          = "QC_relocation"
SEG_ON_RENEWAL           = "ON_renouvellement"
SEG_ON_TURNOVER          = "ON_relocation"

ANCHOR_TAL          = "tal"
ANCHOR_ON_GUIDELINE = "ontario"
ANCHOR_SCHL_QC      = "QC"
ANCHOR_SCHL_ON      = "ON"

REGIME_BY_BUILDING = building_regimes.set_index("sBuilding").regime
is_the_met_exempt = (building_regimes.loc[building_regimes.sState.eq(ONTARIO_LABEL), "regime"]
                     .eq(REGIME_ON_EXEMPT).all())

SEGMENT_ANCHOR = {
    SEG_QC_TAL_RENEWAL:       ANCHOR_TAL,
    SEG_QC_SECTION_F_RENEWAL: ANCHOR_SCHL_QC,
    SEG_QC_TURNOVER:          ANCHOR_SCHL_QC,
    SEG_ON_RENEWAL:           ANCHOR_SCHL_ON if is_the_met_exempt else ANCHOR_ON_GUIDELINE,
    SEG_ON_TURNOVER:          ANCHOR_SCHL_ON,
}

DEFAULT_EXTERNAL = {"tal": TAL_BASE_PCT, "ontario": ONTARIO_GUIDELINE_PCT, "schl": SCHL_RENT_GROWTH_PCT}


def resolve_external(external):
    """Paramètres externes : ceux de la section 7.6, remplacés par ceux passés en argument s'il y en a."""
    return DEFAULT_EXTERNAL if external is None else {**DEFAULT_EXTERNAL, **external}


def assign_segment(pairs_df):
    """Classe chaque paire selon sa province, son régime réglementaire et son type de bail."""
    regime = pairs_df.sBuilding.map(REGIME_BY_BUILDING)
    is_renewal = pairs_df.sRenewal.eq(RENEWAL_FLAG)
    is_ontario = pairs_df.sState.eq(ONTARIO_LABEL)
    conditions = [is_ontario & is_renewal,
                  is_ontario & ~is_renewal,
                  regime.eq(REGIME_QC_TAL) & is_renewal,
                  regime.eq(REGIME_QC_SECTION_F) & is_renewal]
    choices = [SEG_ON_RENEWAL, SEG_ON_TURNOVER, SEG_QC_TAL_RENEWAL, SEG_QC_SECTION_F_RENEWAL]
    return np.select(conditions, choices, default=SEG_QC_TURNOVER)


def external_anchor(segment, year, external):
    """Ancre externe d'un segment pour une année (None si inconnue).
    TAL et Ontario : taux de l'année (publiés avant la plupart des baux).
    SCHL : rapport de l'année précédente, le dernier publié au moment de prévoir."""
    source = SEGMENT_ANCHOR[segment]
    if source == ANCHOR_TAL:
        return external["tal"].get(year)
    if source == ANCHOR_ON_GUIDELINE:
        return external["ontario"].get(year)
    return external["schl"][source].get(year - 1)


def build_pairs(leases_df, price_col):
    """Paires même unité (clé sPropCode + sUnitCode) sur une colonne de prix, avec leur segment."""
    pairs_df = same_unit_growth(leases_df, price_col=price_col)
    pairs_df["segment"] = assign_segment(pairs_df)
    return pairs_df


def segment_weights(pairs_df, year):
    """Part de chaque segment dans les paires d'une année (part de paires, pas d'occupation)."""
    return pairs_df[pairs_df.year == year].segment.value_counts(normalize=True)


def portfolio_growth(pairs_df, year):
    """Hausse combinée d'une année : médiane de chaque segment, pondérée par sa part."""
    medians = pairs_df[pairs_df.year == year].groupby("segment").growth_pct.median()
    weights = segment_weights(pairs_df, year)[medians.index]
    return float((medians * weights).sum() / weights.sum())


def forecast_segment(pairs_df, segment, target_year, spread_weight, anchor_weight, external):
    """Hausse contractuelle prévue d'un segment : mélange entre ancre externe et tendance interne."""
    history_years = range(target_year - LOOKBACK_YEARS, target_year)
    seg_pairs = pairs_df[pairs_df.segment.eq(segment) & pairs_df.year.isin(history_years)]
    internal_by_year = seg_pairs.groupby("year").growth_pct.median()
    if internal_by_year.empty:
        return np.nan

    internal_mean = internal_by_year.mean()
    anchor_target = external_anchor(segment, target_year, external)
    if anchor_target is None:
        return internal_mean

    spreads = [internal_by_year[y] - external_anchor(segment, y, external)
               for y in internal_by_year.index
               if external_anchor(segment, y, external) is not None]
    mean_spread = np.mean(spreads) if spreads else 0.0
    anchored = anchor_target + spread_weight * mean_spread
    return anchor_weight * anchored + (1 - anchor_weight) * internal_mean


def concession_drag(contract_pairs, effective_pairs, segment, target_year, scenario):
    """Effet des concessions (points) = croissance effective - croissance contractuelle.
    bas : effet de la dernière année ; central : moyenne des 3 dernières ; haut : 0 (stabilisation)."""
    if scenario == "haut":
        return 0.0
    years = range(target_year - LOOKBACK_YEARS, target_year)

    def medians_by_year(pairs_df):
        seg_pairs = pairs_df[pairs_df.segment.eq(segment) & pairs_df.year.isin(years)]
        return seg_pairs.groupby("year").growth_pct.median()

    drag_by_year = (medians_by_year(effective_pairs) - medians_by_year(contract_pairs)).dropna()
    if drag_by_year.empty:
        return 0.0
    if scenario == "bas":
        return float(drag_by_year.iloc[-1])
    return float(drag_by_year.mean())


def estimate_growth(contract_pairs, effective_pairs, target_year, spread_weight, anchor_weight,
                    external, scenario=CENTRAL_SCENARIO):
    """Prévision effective de target_year, avec uniquement les paires antérieures à cette année."""
    known_contract  = contract_pairs[contract_pairs.year < target_year]
    known_effective = effective_pairs[effective_pairs.year < target_year]
    weights = segment_weights(known_effective, target_year - 1)

    rows = {}
    for segment in weights.index:
        contract_forecast = forecast_segment(known_contract, segment, target_year,
                                             spread_weight, anchor_weight, external)
        drag = concession_drag(known_contract, known_effective, segment, target_year, scenario)
        rows[segment] = {"contractuel": contract_forecast,
                         "effet_concessions": drag,
                         "effectif": contract_forecast + drag}

    by_segment = pd.DataFrame(rows).T.astype(float).dropna(subset=["effectif"])
    seg_weights = weights[by_segment.index] / weights[by_segment.index].sum()
    by_segment["poids"] = seg_weights
    total = float((by_segment.effectif * seg_weights).sum())
    by_segment = by_segment.round(2)
    by_segment["ancre"] = by_segment.index.map(SEGMENT_ANCHOR)
    return total, by_segment


def backtest_year(contract_pairs, effective_pairs, target_year, spread_weight, anchor_weight, external):
    """Prévision faite au 31 décembre de target_year - 1, comparée au réel et à deux références."""
    predicted, _ = estimate_growth(contract_pairs, effective_pairs, target_year,
                                   spread_weight, anchor_weight, external)
    previous_years = range(target_year - LOOKBACK_YEARS, target_year)
    return {
        "annee": target_year,
        "reel": portfolio_growth(effective_pairs, target_year),
        "modele": predicted,
        "ref_annee_precedente": portfolio_growth(effective_pairs, target_year - 1),
        "ref_moyenne_3_ans": float(np.mean([portfolio_growth(effective_pairs, y) for y in previous_years])),
    }


def run_backtest(contract_pairs, effective_pairs, spread_weight, anchor_weight, external):
    """Backtest sur toutes les années de BACKTEST_YEARS, avec les erreurs absolues."""
    results = pd.DataFrame([backtest_year(contract_pairs, effective_pairs, y,
                                          spread_weight, anchor_weight, external)
                            for y in BACKTEST_YEARS]).set_index("annee")
    for model_col in ["modele", "ref_annee_precedente", "ref_moyenne_3_ans"]:
        results[f"erreur_{model_col}"] = (results[model_col] - results.reel).abs()
    return results


contract_pairs  = build_pairs(leases, CONTRACT_PRICE_COL)
effective_pairs = build_pairs(leases, TARGET_PRICE_COL)

grid_rows = []
for spread_weight in WEIGHT_GRID:
    for anchor_weight in WEIGHT_GRID:
        results = run_backtest(contract_pairs, effective_pairs, spread_weight, anchor_weight, DEFAULT_EXTERNAL)
        grid_rows.append({"poids_ecart": spread_weight, "poids_ancre": anchor_weight,
                          "erreur_moyenne": round(results.erreur_modele.mean(), 2)})
weight_grid = (pd.DataFrame(grid_rows)
               .sort_values(["erreur_moyenne", "poids_ancre"], ascending=[True, False])
               .reset_index(drop=True))
BEST_SPREAD_WEIGHT = float(weight_grid.loc[0, "poids_ecart"])
BEST_ANCHOR_WEIGHT = float(weight_grid.loc[0, "poids_ancre"])


def estimate_2026_details(leases, external=None, scenario=CENTRAL_SCENARIO):
    """Prévision 2026 détaillée par segment, pour un scénario de concessions donné."""
    return estimate_growth(build_pairs(leases, CONTRACT_PRICE_COL), build_pairs(leases, TARGET_PRICE_COL),
                           FORECAST_YEAR, BEST_SPREAD_WEIGHT, BEST_ANCHOR_WEIGHT,
                           resolve_external(external), scenario)


def estimate_2026(leases, asking, external=None):
    """Retourne votre estimation de la hausse de loyer 2026, en pourcentage.

    Définition : croissance annualisée à unité constante du loyer effectif (sRentEffective),
    combinée par segment (province, régime réglementaire, renouvellement ou relocation).
    Choisie parce que le loyer effectif mesure le revenu réellement perçu (voir sections 7.1 et 7.4).
    `asking` n'est pas utilisé dans cette version (voir les limites en 7.7).
    `external` peut remplacer les paramètres de la section 7.6 (clés 'tal', 'ontario', 'schl').
    """
    total, _ = estimate_2026_details(leases, external)
    return round(total, 2)


def backtest(leases, target_year):
    """Exécute votre méthode comme si vous étiez à la fin de target_year - 1,
    et compare le résultat avec ce qui s'est réellement passé en target_year."""
    result = backtest_year(build_pairs(leases, CONTRACT_PRICE_COL), build_pairs(leases, TARGET_PRICE_COL),
                           target_year, BEST_SPREAD_WEIGHT, BEST_ANCHOR_WEIGHT, DEFAULT_EXTERNAL)
    return {key: round(value, 2) for key, value in result.items()}


print("=== Sélection des réglages par backtest ===")
print(weight_grid.to_string(index=False))
print(f"\nRéglages retenus : poids_ecart = {BEST_SPREAD_WEIGHT}, poids_ancre = {BEST_ANCHOR_WEIGHT}")

backtest_results = run_backtest(contract_pairs, effective_pairs,
                                BEST_SPREAD_WEIGHT, BEST_ANCHOR_WEIGHT, DEFAULT_EXTERNAL)
print("\n=== Backtest 2023-2025 (loyer effectif, même unité) ===")
print(backtest_results.round(2).to_string())
print("\nErreur absolue moyenne (points) :")
print(backtest_results.filter(like="erreur_").mean().round(2).to_string())

print("\n=== Vérification suggérée par le notebook de départ ===")
for yr in BACKTEST_YEARS:
    print(yr, backtest(leases, yr))

## 7.8 Prévision 2026

In [ ]:
scenario_results = {}
for scenario in DRAG_SCENARIOS:
    total_2026, segments_2026 = estimate_2026_details(leases, scenario=scenario)
    scenario_results[scenario] = total_2026
    print(f"\n=== Scénario {scenario} : hausse 2026 effective = {total_2026:.2f} % ===")
    print(segments_2026.to_string())

print(f"\nestimate_2026(leases, asking) = {estimate_2026(leases, asking)} %")
print(f"Fourchette : {scenario_results['bas']:.2f} % (bas) à {scenario_results['haut']:.2f} % (haut)")

In [ ]:
try:
    original_section_f_anchor = SEGMENT_ANCHOR[SEG_QC_SECTION_F_RENEWAL]
    SEGMENT_ANCHOR[SEG_QC_SECTION_F_RENEWAL] = ANCHOR_TAL
    estimate_if_section_f_on_tal = estimate_2026(leases, asking)
finally:
    SEGMENT_ANCHOR[SEG_QC_SECTION_F_RENEWAL] = original_section_f_anchor

print(f"Hausse 2026 avec section F ancrée sur le marché (modèle retenu) : {estimate_2026(leases, asking)} %")
print(f"Hausse 2026 si la section F était ancrée sur le TAL (sensibilité) : {estimate_if_section_f_on_tal} %")

fig, (ax_backtest, ax_scenarios) = plt.subplots(1, 2, figsize=(14, 4.5))

backtest_results[["reel", "modele", "ref_annee_precedente", "ref_moyenne_3_ans"]].plot(
    ax=ax_backtest, marker="o", lw=2)
ax_backtest.set_title("Backtest : prévision vs réel (loyer effectif, même unité)")
ax_backtest.set_ylabel("hausse annuelle (%)")
ax_backtest.set_xticks(list(BACKTEST_YEARS))
ax_backtest.grid(alpha=0.3)

scenario_series = pd.Series(scenario_results)[list(DRAG_SCENARIOS)]
ax_scenarios.bar(scenario_series.index, scenario_series.values)
for position, value in enumerate(scenario_series.values):
    ax_scenarios.text(position, value, f"{value:.2f} %", ha="center", va="bottom")
ax_scenarios.set_title("Hausse 2026 effective selon l'évolution des concessions")
ax_scenarios.set_ylabel("hausse 2026 (%)")
ax_scenarios.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

_, central_segments = estimate_2026_details(leases, scenario=CENTRAL_SCENARIO)
building_mix = (effective_pairs[effective_pairs.year == FORECAST_YEAR - 1]
                .groupby("sBuilding").segment.value_counts(normalize=True)
                .unstack(fill_value=0)
                .reindex(columns=central_segments.index, fill_value=0))
building_mix = building_mix.div(building_mix.sum(axis=1), axis=0)
building_forecast = pd.DataFrame({
    "regime": REGIME_BY_BUILDING,
    "hausse_2026_effective_pct": (building_mix @ central_segments.effectif).round(2),
}).sort_values("hausse_2026_effective_pct")
print("\nPrévision 2026 par immeuble (scénario central) :")
print(building_forecast.to_string())

### Prévision par immeuble

La hausse prévue dépend surtout du régime réglementaire de chaque immeuble :
- **Immeubles neufs en section F (Le Carlyle, Westpark) : 4,36 %.** Le TAL ne peut pas y fixer le loyer : les hausses suivent le marché de Montréal.
- **Immeubles encadrés par le TAL (Saint-Elzéar, Lévesque, Daniel-Johnson) : 2,65 à 2,85 %.** Les renouvellements suivent le taux de référence du TAL, qui passe de 5,9 % à 3,1 %.
- **The Met (Ontario, exempté) : 1,88 %.** Malgré l'absence de plafond légal, le marché d'Ottawa est en détente et les concessions y sont importantes : la contrainte vient du marché, pas de la loi.

**Limite :** ce tableau applique les prévisions par segment à la répartition renouvellements/relocations de chaque immeuble en 2025. Il ne s'agit pas d'un modèle distinct par immeuble. Deux immeubles du même régime ne diffèrent donc que par leur proportion de renouvellements, ce qui explique les valeurs identiques du Carlyle et de Westpark.

## 7.9 Estimation finale

**Hausse de loyer 2026 de Collection Équinoxe : 3,25 %** (scénario central)
**Fourchette : 1,77 % à 5,18 %**

**Définition :** croissance annualisée à unité constante du loyer effectif (`sRentEffective`), combinée par segment (province, régime réglementaire, renouvellement ou relocation), pondérée par la part de chaque segment dans les paires de 2025.

**Validation :** au backtest 2023-2025, l'erreur absolue moyenne du modèle est de **1,22 point**, contre 1,55 pour la hausse de l'année précédente et 1,40 pour la moyenne sur 3 ans. La plus grande erreur porte sur 2024, l'année où la part des baux avec concession a bondi de 57 % à 73 %.

**Ce qui explique 2026 :**
- Les renouvellements encadrés par le TAL ralentissent : le taux de référence passe de 5,9 % (2025) à 3,1 % (2026).
- Les relocations et les immeubles neufs (section F) suivent le marché de Montréal, encore en hausse selon la SCHL.
- The Met, exempté de la ligne directrice ontarienne, reste freiné par un marché d'Ottawa en détente.
- Les concessions retirent environ 2 points à la hausse contractuelle dans le scénario central.

**La principale incertitude, ce sont les concessions.** L'écart entre le scénario bas (1,77 %) et le scénario haut (5,18 %) vient entièrement de l'hypothèse sur leur évolution. Le loyer affiché, lui, est relativement prévisible.

**Sensibilité au régime de la section F.** Le Carlyle et Westpark sont traités comme des immeubles neufs dont les renouvellements suivent le marché (SCHL Montréal), puisque le TAL ne peut pas y fixer le loyer. Ce choix est juridiquement fondé, mais il ne repose que sur 2 années de données, et en 2025 leurs renouvellements ont augmenté presque autant que ceux des immeubles encadrés (3,01 % contre 2,93 % en loyer effectif). Si on les ancrait plutôt sur le TAL, la prévision passerait de **3,25 % à 2,74 %**. L'écart (0,51 point) reste inférieur à l'erreur moyenne du backtest (1,22 point) : l'hypothèse influence le résultat sans le remettre en cause.

**Hypothèses :**
1. Chaque segment suit, à parts égales, son ancre externe et sa tendance des 3 dernières années (réglage choisi par backtest).
2. Les concessions continuent de peser au rythme moyen de 2023-2025 (scénario central).
3. The Met est exempté de la ligne directrice ontarienne (premier bail en 2023, après le 15 novembre 2018).
4. Le Carlyle et Westpark relèvent de la section F (immeubles de 5 ans ou moins).
5. La répartition des segments en 2026 ressemble à celle de 2025.

**Limites :** l'âge des immeubles est déduit du premier bail dans nos données ; les segments section F et The Met n'ont que 2 années d'historique ; les réglages sont choisis sur les années du backtest ; la table `asking` n'est pas exploitée ; l'extrait ne permet pas d'utiliser l'occupation ; les régimes sont établis selon l'âge des immeubles en 2026 et appliqués tels quels dans le backtest (Daniel-Johnson, par exemple, avait moins de 5 ans en 2023).

## Références

### Données publiques
- Tribunal administratif du logement, communiqués sur le calcul de l'augmentation des loyers 2022 et 2023 — tal.gouv.qc.ca
- Le Devoir, « Les loyers devraient augmenter d'au moins 3,1 % en 2026, selon le TAL », janvier 2026
- Gouvernement de l'Ontario, ligne directrice sur l'augmentation des loyers — ontario.ca/page/rent-increase-guideline
- SCHL, Rapport sur le marché locatif, éditions 2022 à 2025 — cmhc-schl.gc.ca
- SCHL, Mise à jour sur le marché locatif de mi-année 2026
- La Presse, « Un marché difficile malgré une croissance moindre des loyers », décembre 2024
- Code civil du Québec, art. 1955 (section F du bail, immeubles de 5 ans ou moins)

### Outils d'IA
- Claude (Anthropic) : explication du notebook de départ, rédaction de cellules Markdown, recherche de sources publiques.